In [1]:
## IMPORT LIBRARIES

In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

## LOAD DATASET 

In [4]:
cols = ['lettr','x-box','y-box','width','height','onpix','x-bar','y-bar',
        'x2bar','y2bar','xybar','x2ybr','xy2br','x-ege','xegvy','y-ege','yegvx']

df = pd.read_csv("letter-recognition.data", header=None, names=cols)

##  PREPROCESSING

In [5]:
X = df.drop('lettr', axis=1).values # .values Converts dataframe into numpy array
y = df['lettr'].values

# Encode labels (A–Z → 0–25)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# One-hot encoding
y_cat = to_categorical(y_encoded, num_classes=26)  # Converts numbers into binary vectors to remove number order problem

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42
)

# Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



##  BUILD MODEL

In [6]:
model = Sequential()

model.add(Dense(128, activation='relu', input_dim=16))
model.add(Dense(64, activation='relu'))
model.add(Dense(26, activation='softmax'))

C:\Users\omsod\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Train Model

In [8]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,   # data is divided into small groups (batches) and Each batch has 64 samples
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5267 - loss: 1.7738 - val_accuracy: 0.7131 - val_loss: 1.0147
Epoch 2/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7651 - loss: 0.8333 - val_accuracy: 0.7981 - val_loss: 0.7225
Epoch 3/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8191 - loss: 0.6316 - val_accuracy: 0.8269 - val_loss: 0.5955
Epoch 4/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8492 - loss: 0.5217 - val_accuracy: 0.8644 - val_loss: 0.4901
Epoch 5/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8692 - loss: 0.4442 - val_accuracy: 0.8781 - val_loss: 0.4298
Epoch 6/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8865 - loss: 0.3862 - val_accuracy: 0.8863 - val_loss: 0.3900
Epoch 7/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8980 - loss: 0.3440 - val_accuracy: 0.9031 - val_loss: 0.3439
Epoch 8/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9086 - loss: 0.3056 - val_accuracy: 0.

## Predictions

In [9]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9548 - loss: 0.1509  
Test Accuracy: 0.9547500014305115


##  PREDICT SAMPLE

In [10]:
sample = np.array([[2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8]])
sample = scaler.transform(sample)

pred = model.predict(sample)
pred_class = np.argmax(pred) # Finds index of highest value

print("Predicted Letter:", le.classes_[pred_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted Letter: T
